# 4. Is the estimate trustworthy? Regression and ML vs our model
We have no ground-truth crowding, so we test whether the pattern we estimate is **stable and predictable over time**. Time-based hold-out with three 7-day windows (never a random split, which would leak):
- features from week 2 -> predict week 3 for every route-stretch-time cell;
- trained models (linear regression, gradient boosting) first learn week 1 -> week 2.
Only cells with enough tickets in week 3 count as reliable truth.

In [ ]:
import sys, os
sys.path.append("..")                       # lets the notebook import config.py and the crowd/ folder
# os.environ["DATA_MODE"] = "synthetic"     # <- uncomment to try everything on the demo data
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import config
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
P = config.PROC; print("mode:", config.MODE, "| processed folder:", P)

In [ ]:
import json
from crowd import validate
tx = pd.read_parquet(P / 'tickets_resolved.parquet'); pat = pd.read_parquet(P / 'patterns.parquet'); hourly = pd.read_parquet(P / 'route_hourly.parquet')
attrs = pd.read_parquet(P / 'route_attrs.parquet'); cells = pd.read_parquet(P / 'crowd_segments.parquet'); cal = json.load(open(P / 'calibration.json'))
d = tx.groupby('date').agg(pax=('pax', 'sum'), dow=('dow', 'first')).reset_index(); wk = d[d.dow < 6].pax; med = wk.median(); mad = (wk - med).abs().median()
excl = d[(d.dow < 6) & ((d.pax - med).abs() > 3.5 * 1.4826 * mad)].date.tolist(); print('excluded days:', excl)
M, tab, sens, ins = validate.run_validation(tx, hourly, pat, attrs, cells, cal, excl, config.OUT)
tab

**How to read the table** (every row predicts the ticketed load per trip of week-3 cells):
- **MAE / RMSE**: average error, lower is better. **R²**: share of variation explained, 1 is perfect, below 0 is worse than predicting the overall mean. **Spearman**: do we rank busy vs quiet cells correctly?
- *City-wide* and *route-level average* are simple baselines. *Linear regression* is the classic ML baseline: it adds effects (route + band + day + position) and cannot capture that a stretch is busy only at certain hours on a certain route. *Gradient boosting* learns those interactions. Our *shrinkage model* uses each cell's own history, pulled toward the route average when evidence is thin.

In [ ]:
sns.barplot(data=tab.reset_index(), y='index', x='R2', color='#12263F'); plt.xlabel('R² on week 3 (higher is better)'); plt.ylabel(''); plt.show()
pd.Series({k: v for k, v in M.items() if k != 'holdout'}).to_frame('value')

## The ML suite (also served by `/api/ml/insights`)
Four regressors predict next-week load per cell; two classifiers predict the crowd level. All share one feature set (own history, route average, city average, band, day type, position along the route, route length, scheduled trips, AC flag). The best regressor on **thinly observed** cells becomes the *fallback* that fills stretches with few tickets.

In [ ]:
reg = pd.DataFrame(ins['regression_models']).set_index('name'); clf = pd.DataFrame(ins['classification_models']).set_index('name')
display(reg); display(clf); print('thin-cell comparison:', ins['thin_cell_comparison']); print('active fallback:', ins['active_fallback_model'])
cm = ins['confusion_matrix']; sns.heatmap(pd.DataFrame(cm['matrix'], index=cm['labels'], columns=cm['labels']), annot=True, fmt='d', cmap='Blues'); plt.title('Confusion matrix: ' + cm['model']); plt.xlabel('predicted'); plt.ylabel('actual'); plt.show()
pd.DataFrame(ins['feature_importance']).set_index('feature').weight.plot.barh(title='Permutation importance (gradient boosting)'); plt.show()
print('Lasso zeroed these features:', ins['lasso_zeroed_features'])

In [ ]:
sens   # how many cells would be labelled crowded if the coverage-factor assumption were wrong by 0.5x ... 2x

**Be honest in the technical note:** these tests show the estimate is consistent and predictable from past weeks, and that the pattern follows commute peaks. They do **not** prove the absolute 'seats / standing / crowded' labels, because no ground-truth crowding exists in the data; those depend on the stated capacity and coverage assumptions.